# ETL Notebook 
This notebook documents my step-by-step thinking and justification behind my decisions for building the ETL pipeline.

In [1]:
from pathlib import Path
import pandas as pd

Reading the data from .csv files and storing into pandas data frames

In [2]:
PMDATA_DIR = Path("data/pmdata")

injury_frames, srpe_frames, wellness_frames = [], [], []

# p[0-9][0-9] matches p01..p16 only — NOT participant-overview.xlsx
# sorted() keeps them in order
for athlete_dir in sorted(PMDATA_DIR.glob("p[0-9][0-9]")):
    source_id = athlete_dir.name # e.g. "p01"
    pmsys = athlete_dir / "pmsys"

    # srpe + wellness are present for every athlete
    srpe = pd.read_csv(pmsys / "srpe.csv", parse_dates=["end_date_time"])
    srpe.insert(0, "source_id", source_id)
    srpe_frames.append(srpe)

    wellness = pd.read_csv(pmsys / "wellness.csv", parse_dates=["effective_time_frame"])
    wellness.insert(0, "source_id", source_id)
    wellness_frames.append(wellness)

    # guard: p08 has no injury.csv
    injury_path = pmsys / "injury.csv"
    if injury_path.exists():
        injury = pd.read_csv(injury_path, parse_dates=["effective_time_frame"])
        injury.insert(0, "source_id", source_id)
        injury_frames.append(injury)
    else:
        print(f"skipping injury for {source_id}: {injury_path} not found")

injury = pd.concat(injury_frames, ignore_index=True)
srpe = pd.concat(srpe_frames, ignore_index=True)
wellness = pd.concat(wellness_frames, ignore_index=True)

for name, df in {"injury": injury, "srpe": srpe, "wellness": wellness}.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols, "
          f"{df['source_id'].nunique()} athletes")

skipping injury for p08: data\pmdata\p08\pmsys\injury.csv not found
injury: 225 rows x 3 cols, 15 athletes
srpe: 783 rows x 5 cols, 16 athletes
wellness: 1747 rows x 10 cols, 16 athletes


Sanity check on participant_overview df

In [3]:
participant_overview = pd.read_excel(PMDATA_DIR / "participant-overview.xlsx", header=1)
participant_overview

,Participant ID,Age,Height,Gender,A or B person,Max heart rate,Date,Minutes,Seconds,Stride walk,Stride run
0,p01,48,195,male,A,182,2019-11-26 00:00:00,29,33,80.90,102.9
1,p02,60,180,male,A,169,2019-12-15 00:00:00,23,51,74.70,92.4
2,p03,25,184,male,A,157,2019-12-30 00:00:00,33,22,NaN,NaN
3,p04,26,163,female,A,195,2019-11-19 00:00:00,22,13,67.30,110.2
4,p05,35,176,male,A,184,2019-12-23 00:00:00,32,40,73.00,94.3
5,p06,42,179,male,B,181,2019-12-01 00:00:00,23,19,73.04,97.6
6,p07,26,177,male,B,NaN,2019-11-19 00:00:00,19,40,73.50,119.5
7,p08,27,186,male,B,200,2019-11-28 00:00:00,18,47,77.20,103.6
8,p09,26,180,male,B,183,2020-01-07 00:00:00,35,6,74.70,109.9
9,p10,38,179,female,B,197,2019-12-08 00:00:00,28,10,73.09,102.3


Decided to only retain demographic information about each athlete:

In [4]:
participant_overview = participant_overview.drop(columns=['Date', 'Minutes', 'Seconds', 'Stride walk', 'Stride run'])
participant_overview

,Participant ID,Age,Height,Gender,A or B person,Max heart rate
0,p01,48,195,male,A,182
1,p02,60,180,male,A,169
2,p03,25,184,male,A,157
3,p04,26,163,female,A,195
4,p05,35,176,male,A,184
5,p06,42,179,male,B,181
6,p07,26,177,male,B,NaN
7,p08,27,186,male,B,200
8,p09,26,180,male,B,183
9,p10,38,179,female,B,197


I noticed p08 had no injury.csv file and wanted to inspect how many injuries were being reported and how many of them were empty per athlete:

In [5]:
injury["is_empty"] = injury["injuries"].isin([{}, "{}"])   # dict vs string, see below
print(injury.groupby("source_id")["is_empty"].agg(["sum", "count"]))

           sum  count
source_id            
p01         23     24
p02          4      5
p03         11     11
p04         14     17
p05          0     10
p06         15     15
p07         13     15
p09         20     20
p10         12     12
p11          1      4
p12          1     49
p13         16     16
p14          7     10
p15          0      6
p16         11     11


This tells us that there aren't clean periodic check-ins. Otherwise we would have "{}" - empty injury check-ins on days where the athlete is healthy and injury free.  

In [7]:
p12 = injury[injury["source_id"] == "p12"]
p12[p12["injuries"] != "{}"][["effective_time_frame", "injuries"]].head(20)

,effective_time_frame,injuries
133,2019-11-15 20:01:20.647000+00:00,{'right_knee': 'minor'}
135,2019-11-18 04:50:00.666000+00:00,{'right_knee': 'minor'}
136,2019-11-18 06:01:40.574000+00:00,{'right_knee': 'minor'}
137,2019-11-18 19:26:24.936000+00:00,{'left_shoulder': 'minor'}
138,2019-11-19 05:49:39.091000+00:00,"{'left_leg': 'minor', 'left_shoulder': 'minor'..."
139,2019-11-20 05:58:09.329000+00:00,"{'left_leg': 'minor', 'left_shoulder': 'minor'..."
140,2019-11-20 20:03:10.579000+00:00,"{'left_leg': 'minor', 'right_knee': 'minor'}"
141,2019-11-21 20:59:32.959000+00:00,"{'left_leg': 'minor', 'right_knee': 'minor'}"
142,2019-11-22 22:35:26.660000+00:00,"{'left_leg': 'minor', 'right_knee': 'minor'}"
143,2019-11-24 10:10:14.803000+00:00,{'right_knee': 'minor'}
